In [1]:
pip install flask


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:

#creatng folders
import os
os.makedirs("college_erp/templates",exist_ok=True)

os.makedirs("college_erp/static",exist_ok=True)

print("collge erp folders created successfully")

    

collge erp folders created successfully


In [5]:

#connection to database 
import sqlite3
database="college_erp/college_erp.db"

connection=sqlite3.connect(database)

cursor=connection.cursor()
print("database has connected")
connection.close()


database has connected


In [9]:
import sqlite3

connection = sqlite3.connect(
    "college_erp/college_erp.db"
)

cursor = connection.cursor()


# Student table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    usn TEXT NOT NULL,

    name TEXT NOT NULL,

    department TEXT NOT NULL,

    semester TEXT NOT NULL,

    email TEXT,

    phone TEXT

)
""")


# Faculty table
cursor.execute("""
CREATE TABLE IF NOT EXISTS faculty (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    name TEXT NOT NULL,

    department TEXT NOT NULL,

    subject TEXT NOT NULL,

    email TEXT,

    phone TEXT

)
""")


# Attendance table
cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    student_id INTEGER,

    date TEXT,

    status TEXT

)
""")


# Marks table
cursor.execute("""
CREATE TABLE IF NOT EXISTS marks (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    student_id INTEGER,

    subject TEXT,

    marks INTEGER

)
""")


connection.commit()

connection.close()

print("All database tables created successfully!")

All database tables created successfully!


In [3]:
app_code = """
from flask import Flask, render_template, request, redirect, session
import sqlite3

app = Flask(__name__)

# Secret key is required for login sessions
app.secret_key = "college_erp_secret"

# SQLite database
DATABASE = "college_erp.db"


# -----------------------------
# Database connection
# -----------------------------

def get_db():
    connection = sqlite3.connect(DATABASE)
    connection.row_factory = sqlite3.Row
    return connection


# -----------------------------
# Create database tables
# -----------------------------

def create_tables():

    connection = get_db()
    cursor = connection.cursor()

    # Student table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            usn TEXT NOT NULL,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            semester TEXT NOT NULL,
            email TEXT,
            phone TEXT
        )
    ''')

    # Faculty table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS faculty (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            department TEXT NOT NULL,
            subject TEXT NOT NULL,
            email TEXT,
            phone TEXT
        )
    ''')

    # Attendance table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS attendance (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            student_id INTEGER,
            date TEXT,
            status TEXT
        )
    ''')

    # Marks table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS marks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            student_id INTEGER,
            subject TEXT,
            marks INTEGER
        )
    ''')

    connection.commit()
    connection.close()


# -----------------------------
# Login page
# -----------------------------

@app.route("/", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        username = request.form["username"]
        password = request.form["password"]

        # Simple admin login
        if username == "admin" and password == "admin123":

            session["logged_in"] = True

            return redirect("/dashboard")

        return render_template(
            "login.html",
            error="Invalid username or password"
        )

    return render_template("login.html")


# -----------------------------
# Logout
# -----------------------------

@app.route("/logout")
def logout():

    session.clear()

    return redirect("/")


# -----------------------------
# Dashboard
# -----------------------------

@app.route("/dashboard")
def dashboard():

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    # Count students
    students = connection.execute(
        "SELECT COUNT(*) AS total FROM students"
    ).fetchone()["total"]

    # Count faculty
    faculty = connection.execute(
        "SELECT COUNT(*) AS total FROM faculty"
    ).fetchone()["total"]

    # Count attendance records
    attendance = connection.execute(
        "SELECT COUNT(*) AS total FROM attendance"
    ).fetchone()["total"]

    # Count marks records
    marks = connection.execute(
        "SELECT COUNT(*) AS total FROM marks"
    ).fetchone()["total"]

    connection.close()

    return render_template(
        "dashboard.html",
        students=students,
        faculty=faculty,
        attendance=attendance,
        marks=marks
    )


# -----------------------------
# Students
# -----------------------------

@app.route("/students", methods=["GET", "POST"])
def students():

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    # Add student
    if request.method == "POST":

        usn = request.form["usn"]
        name = request.form["name"]
        department = request.form["department"]
        semester = request.form["semester"]
        email = request.form["email"]
        phone = request.form["phone"]

        connection.execute('''
            INSERT INTO students
            (usn, name, department, semester, email, phone)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (
            usn,
            name,
            department,
            semester,
            email,
            phone
        ))

        connection.commit()

    # Search
    search = request.args.get("search", "")

    if search:

        student_list = connection.execute('''
            SELECT * FROM students
            WHERE name LIKE ?
            OR usn LIKE ?
            OR department LIKE ?
            ORDER BY id DESC
        ''', (
            "%" + search + "%",
            "%" + search + "%",
            "%" + search + "%"
        )).fetchall()

    else:

        student_list = connection.execute(
            "SELECT * FROM students ORDER BY id DESC"
        ).fetchall()

    connection.close()

    return render_template(
        "students.html",
        students=student_list,
        search=search
    )


# -----------------------------
# Delete student
# -----------------------------

@app.route("/delete_student/<int:id>")
def delete_student(id):

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    connection.execute(
        "DELETE FROM students WHERE id = ?",
        (id,)
    )

    connection.commit()
    connection.close()

    return redirect("/students")


# -----------------------------
# Faculty
# -----------------------------

@app.route("/faculty", methods=["GET", "POST"])
def faculty():

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    if request.method == "POST":

        name = request.form["name"]
        department = request.form["department"]
        subject = request.form["subject"]
        email = request.form["email"]
        phone = request.form["phone"]

        connection.execute('''
            INSERT INTO faculty
            (name, department, subject, email, phone)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            name,
            department,
            subject,
            email,
            phone
        ))

        connection.commit()

    faculty_list = connection.execute(
        "SELECT * FROM faculty ORDER BY id DESC"
    ).fetchall()

    connection.close()

    return render_template(
        "faculty.html",
        faculty=faculty_list
    )


# -----------------------------
# Delete faculty
# -----------------------------

@app.route("/delete_faculty/<int:id>")
def delete_faculty(id):

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    connection.execute(
        "DELETE FROM faculty WHERE id = ?",
        (id,)
    )

    connection.commit()
    connection.close()

    return redirect("/faculty")


# -----------------------------
# Attendance
# -----------------------------

@app.route("/attendance", methods=["GET", "POST"])
def attendance():

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    if request.method == "POST":

        student_id = request.form["student_id"]
        date = request.form["date"]
        status = request.form["status"]

        connection.execute('''
            INSERT INTO attendance
            (student_id, date, status)
            VALUES (?, ?, ?)
        ''', (
            student_id,
            date,
            status
        ))

        connection.commit()

    student_list = connection.execute(
        "SELECT * FROM students"
    ).fetchall()

    attendance_list = connection.execute('''
        SELECT
            attendance.id,
            students.usn,
            students.name,
            attendance.date,
            attendance.status
        FROM attendance
        JOIN students
        ON attendance.student_id = students.id
        ORDER BY attendance.id DESC
    ''').fetchall()

    connection.close()

    return render_template(
        "attendance.html",
        students=student_list,
        attendance=attendance_list
    )


# -----------------------------
# Marks
# -----------------------------

@app.route("/marks", methods=["GET", "POST"])
def marks():

    if "logged_in" not in session:
        return redirect("/")

    connection = get_db()

    if request.method == "POST":

        student_id = request.form["student_id"]
        subject = request.form["subject"]
        marks_value = request.form["marks"]

        connection.execute('''
            INSERT INTO marks
            (student_id, subject, marks)
            VALUES (?, ?, ?)
        ''', (
            student_id,
            subject,
            marks_value
        ))

        connection.commit()

    student_list = connection.execute(
        "SELECT * FROM students"
    ).fetchall()

    marks_list = connection.execute('''
        SELECT
            marks.id,
            students.usn,
            students.name,
            marks.subject,
            marks.marks
        FROM marks
        JOIN students
        ON marks.student_id = students.id
        ORDER BY marks.id DESC
    ''').fetchall()

    connection.close()

    return render_template(
        "marks.html",
        students=student_list,
        marks=marks_list
    )


# -----------------------------
# Start application
# -----------------------------

if __name__ == "__main__":

    create_tables()

    app.run(
        debug=False,
        use_reloader=False,
        port=5000
    )
"""


with open("college_erp/app.py", "w", encoding="utf-8") as file:
    file.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [4]:
login_html="""
<!doctype html>
<html>
<head>
<title>COLLEGE ERP LOGIN</title>
<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">
</head>

<body class="login-page">
<div class="login-box">
<h1>COLLEGE ERP </h1>
<h3>ADMIN LOGIN</h3>
{% if error %}
<p class="error">

{{ error }}

</p>

{% endif %}


<form method="POST">

<input
type="text"
name="username"
placeholder="Username"
required>


<input
type="password"
name="password"
placeholder="Password"
required>


<button type="submit">

Login

</button>


</form>


<p>

Username: <b>admin</b>

<br>

Password: <b>admin123</b>

</p>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/login.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(login_html)


print("login.html created successfully!")


login.html created successfully!


In [5]:
#dashboard.html file

dashboard_html = """

<!DOCTYPE html>

<html>

<head>

<title>College ERP Dashboard</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<nav>

<h2>🎓 College ERP</h2>


<div>

<a href="/dashboard">
Dashboard
</a>

<a href="/students">
Students
</a>

<a href="/faculty">
Faculty
</a>

<a href="/attendance">
Attendance
</a>

<a href="/marks">
Marks
</a>

<a href="/logout">
Logout
</a>

</div>

</nav>


<div class="container">


<h1>Dashboard</h1>

<p>
Welcome to College ERP
</p>


<div class="cards">


<div class="card">

<h2>{{ students }}</h2>

<p>
Total Students
</p>

</div>


<div class="card">

<h2>{{ faculty }}</h2>

<p>
Total Faculty
</p>

</div>


<div class="card">

<h2>{{ attendance }}</h2>

<p>
Attendance Records
</p>

</div>


<div class="card">

<h2>{{ marks }}</h2>

<p>
Marks Records
</p>

</div>


</div>


<div class="welcome">

<h2>
College Management System
</h2>

<p>

Manage students, faculty,
attendance and marks.

</p>

</div>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/dashboard.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(dashboard_html)


print("dashboard.html created successfully!")

dashboard.html created successfully!


In [6]:
#student.html file

students_html = """

<!DOCTYPE html>

<html>

<head>

<title>Students</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<nav>

<h2>🎓 College ERP</h2>

<div>

<a href="/dashboard">Dashboard</a>

<a href="/students">Students</a>

<a href="/faculty">Faculty</a>

<a href="/attendance">Attendance</a>

<a href="/marks">Marks</a>

<a href="/logout">Logout</a>

</div>

</nav>


<div class="container">


<h1>👨‍🎓 Student Management</h1>


<div class="form-box">

<h2>Add Student</h2>


<form method="POST">


<input
name="usn"
placeholder="USN"
required>


<input
name="name"
placeholder="Student Name"
required>


<input
name="department"
placeholder="Department"
required>


<input
name="semester"
placeholder="Semester"
required>


<input
name="email"
placeholder="Email">


<input
name="phone"
placeholder="Phone">


<button>

Add Student

</button>


</form>

</div>


<div class="search-box">

<form method="GET">


<input
name="search"
value="{{ search }}"
placeholder="Search student">


<button>

Search

</button>


<a href="/students">

Clear

</a>


</form>

</div>


<h2>Student List</h2>


<table>


<tr>

<th>ID</th>

<th>USN</th>

<th>Name</th>

<th>Department</th>

<th>Semester</th>

<th>Email</th>

<th>Phone</th>

<th>Action</th>

</tr>


{% for student in students %}


<tr>

<td>{{ student.id }}</td>

<td>{{ student.usn }}</td>

<td>{{ student.name }}</td>

<td>{{ student.department }}</td>

<td>{{ student.semester }}</td>

<td>{{ student.email }}</td>

<td>{{ student.phone }}</td>


<td>

<a
class="delete"
href="/delete_student/{{ student.id }}">

Delete

</a>

</td>

</tr>


{% endfor %}


</table>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/students.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(students_html)


print("students.html created successfully!")

students.html created successfully!


In [7]:
#faculties file
faculty_html = """

<!DOCTYPE html>

<html>

<head>

<title>Faculty</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<nav>

<h2>🎓 College ERP</h2>

<div>

<a href="/dashboard">Dashboard</a>

<a href="/students">Students</a>

<a href="/faculty">Faculty</a>

<a href="/attendance">Attendance</a>

<a href="/marks">Marks</a>

<a href="/logout">Logout</a>

</div>

</nav>


<div class="container">


<h1>👩‍🏫 Faculty Management</h1>


<div class="form-box">


<h2>Add Faculty</h2>


<form method="POST">


<input
name="name"
placeholder="Faculty Name"
required>


<input
name="department"
placeholder="Department"
required>


<input
name="subject"
placeholder="Subject"
required>


<input
name="email"
placeholder="Email">


<input
name="phone"
placeholder="Phone">


<button>

Add Faculty

</button>


</form>


</div>


<h2>Faculty List</h2>


<table>


<tr>

<th>ID</th>

<th>Name</th>

<th>Department</th>

<th>Subject</th>

<th>Email</th>

<th>Phone</th>

<th>Action</th>

</tr>


{% for teacher in faculty %}


<tr>

<td>{{ teacher.id }}</td>

<td>{{ teacher.name }}</td>

<td>{{ teacher.department }}</td>

<td>{{ teacher.subject }}</td>

<td>{{ teacher.email }}</td>

<td>{{ teacher.phone }}</td>


<td>

<a
class="delete"
href="/delete_faculty/{{ teacher.id }}">

Delete

</a>

</td>

</tr>


{% endfor %}


</table>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/faculty.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(faculty_html)


print("faculty.html created successfully!")

faculty.html created successfully!


In [8]:
attendance_html = """

<!DOCTYPE html>

<html>

<head>

<title>Attendance</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<nav>

<h2>🎓 College ERP</h2>

<div>

<a href="/dashboard">Dashboard</a>

<a href="/students">Students</a>

<a href="/faculty">Faculty</a>

<a href="/attendance">Attendance</a>

<a href="/marks">Marks</a>

<a href="/logout">Logout</a>

</div>

</nav>


<div class="container">


<h1>📅 Attendance Management</h1>


<div class="form-box">


<h2>Mark Attendance</h2>


<form method="POST">


<select
name="student_id"
required>


<option value="">

Select Student

</option>


{% for student in students %}


<option value="{{ student.id }}">

{{ student.usn }} -
{{ student.name }}

</option>


{% endfor %}


</select>


<input
type="date"
name="date"
required>


<select name="status">


<option value="Present">

Present

</option>


<option value="Absent">

Absent

</option>


</select>


<button>

Save Attendance

</button>


</form>


</div>


<h2>Attendance Records</h2>


<table>


<tr>

<th>ID</th>

<th>USN</th>

<th>Student</th>

<th>Date</th>

<th>Status</th>

</tr>


{% for record in attendance %}


<tr>

<td>{{ record.id }}</td>

<td>{{ record.usn }}</td>

<td>{{ record.name }}</td>

<td>{{ record.date }}</td>

<td>{{ record.status }}</td>

</tr>


{% endfor %}


</table>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/attendance.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(attendance_html)


print("attendance.html created successfully!")

attendance.html created successfully!


In [9]:
#marks .html 
marks_html = """

<!DOCTYPE html>

<html>

<head>

<title>Marks</title>

<link rel="stylesheet"
href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<nav>

<h2>🎓 College ERP</h2>

<div>

<a href="/dashboard">Dashboard</a>

<a href="/students">Students</a>

<a href="/faculty">Faculty</a>

<a href="/attendance">Attendance</a>

<a href="/marks">Marks</a>

<a href="/logout">Logout</a>

</div>

</nav>


<div class="container">


<h1>📊 Academic Marks</h1>


<div class="form-box">


<h2>Add Marks</h2>


<form method="POST">


<select
name="student_id"
required>


<option value="">

Select Student

</option>


{% for student in students %}


<option value="{{ student.id }}">

{{ student.usn }} -
{{ student.name }}

</option>


{% endfor %}


</select>


<input
name="subject"
placeholder="Subject"
required>


<input
name="marks"
type="number"
min="0"
max="100"
placeholder="Marks"
required>


<button>

Add Marks

</button>


</form>


</div>


<h2>Marks Records</h2>


<table>


<tr>

<th>ID</th>

<th>USN</th>

<th>Student</th>

<th>Subject</th>

<th>Marks</th>

</tr>


{% for record in marks %}


<tr>

<td>{{ record.id }}</td>

<td>{{ record.usn }}</td>

<td>{{ record.name }}</td>

<td>{{ record.subject }}</td>

<td>{{ record.marks }}</td>

</tr>


{% endfor %}


</table>


</div>


</body>

</html>

"""


with open(

    "college_erp/templates/marks.html",

    "w",

    encoding="utf-8"

) as file:

    file.write(marks_html)


print("marks.html created successfully!")

marks.html created successfully!


In [10]:
css_code="""
*{
margin:0;
padding:0;
box-sizing:border-box;
}

body{
margin:0;
font-family:Arial;
background:whitesmoke;
}

nav{
background:navy;
color:white;
padding:15px 34px;
display:flex;
justify-content:space-between;
align-items:center;
flex-wrap:wrap;
}

nav a{

color:white;
text-decoration:none;
margin:8px;
}

.container{

width:90%;
max-width:1200px;
margin:30px;

}

.cards {

    display: grid;

    grid-template-columns:
    repeat(auto-fit, minmax(200px, 1fr));

    gap: 20px;
}


.card {

    background: white;

    padding: 25px;

    text-align: center;

    border-radius: 10px;

    box-shadow:
    0 2px 8px lightgray;
}


.card h2 {

    font-size: 35px;
}


.form-box {

    background: white;

    padding: 25px;

    margin: 20px 0;

    border-radius: 10px;
}


input,
select {

    padding: 12px;

    margin: 5px;

    border: 1px solid lightgray;

    border-radius: 5px;
}


button {

    padding: 12px 20px;

    background: navy;

    color: white;

    border: none;

    border-radius: 5px;

    cursor: pointer;
}


table {

    width: 100%;

    border-collapse: collapse;

    background: white;
}


th,
td {

    padding: 12px;

    border: 1px solid lightgray;
}


th {

    background: navy;

    color: white;
}


.delete {

    color: red;

    text-decoration: none;
}


.login-page {

    background: navy;

    min-height: 100vh;

    display: flex;

    justify-content: center;

    align-items: center;
}


.login-box {

    background: white;

    padding: 40px;

    width: 350px;

    border-radius: 15px;

    text-align: center;
}


.login-box input {

    display: block;

    width: 100%;

    margin: 15px 0;
}


.login-box button {

    width: 100%;
}


.error {

    color: red;
}


@media(max-width:700px) {

    nav {

        flex-direction: column;
    }

    input,
    select {

        width: 100%;
    }

}

"""


with open(

    "college_erp/static/style.css",

    "w",

    encoding="utf-8"

) as file:

    file.write(css_code)


print("style.css created successfully!")

style.css created successfully!


In [13]:
#checking the project done

import os

for root,folders,files in os.walk("college_erp"):
    print("\n",root)
    for file in files:
        print("",file)


 college_erp
 app.py
 college_erp.db

 college_erp\static
 style.css

 college_erp\templates
 attendance.html
 dashboard.html
 faculty.html
 login.html
 marks.html
 students.html


In [14]:
import subprocess
import sys

process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd="college_erp"
)

print("College ERP is running!")
print("Open: http://127.0.0.1:5000")

College ERP is running!
Open: http://127.0.0.1:5000
